# Random Forest classification on text & handwriting corpus

Train a **TF-IDF + Random Forest** classifier on the smol-doc-analyzer synthetic insurance document corpus:

- **Typed text** — clean Stage A documents (`data/synthetic/documents/`)
- **Handwriting / OCR surface** — character-garbled noisy variants (`data/synthetic/noisy/`) that approximate OCR'd handwriting and scanned forms

**Targets**

1. Primary: predict `document_type` (ACORD-inspired taxonomy)
2. Secondary: predict surface style (`typed` vs `handwriting_ocr`)

This notebook is a lightweight classical baseline alongside the DeBERTa / ViT deep classifiers.

**Multilayer WandB suite** (section 9) runs the same pipeline as the CLI:

0. corpus profile → 1. capacity sweep → 2. dual heads → 3. surface slices →
4. confidence/ECE → 5. feature importance / confusion pairs

Logged under project `smol-doc-analyzer`, run name `rf-notebook-multilayer`.
Corpus seeding does **not** open a WandB run from this notebook.

## 1. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay

# Allow running the notebook from notebooks/ or repo root
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.classification.random_forest import (
    DEFAULT_PRESET_NAMES,
    SURFACE_HANDWRITING_OCR,
    SURFACE_TYPED,
    assign_split_column,
    build_document_type_pipeline,
    ensure_seed_corpus,
    evaluate_classifier,
    load_text_handwriting_corpus,
    save_random_forest_bundle,
    top_tfidf_feature_importances,
    write_predictions_jsonl,
)
from src.classification.train_random_forest import train as train_rf_multilayer
from src.utils.config import Config
from src.utils.wandb_utils import load_wandb_settings

pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")
cfg = Config.load()
print("repo:", REPO_ROOT)
print("documents dir:", cfg.document_output_dir)
print("noisy dir:", cfg.noisy_output_dir)

## 2. Ensure corpus (text + handwriting/OCR noise)

If synthetic documents are missing, regenerate a seeded corpus (template mode; no API key required).

In [ ]:
SEED_N = 240
SEED = 42

# log_wandb=False: don't open a seed_pipeline WandB run from this notebook
corpus_paths = ensure_seed_corpus(n=SEED_N, seed=SEED, log_wandb=False)
display(pd.Series(corpus_paths, name="path").to_frame())

## 3. Load typed + handwriting/OCR documents

In [ ]:
frame = load_text_handwriting_corpus()
frame = assign_split_column(frame)

print(f"rows: {len(frame):,}  |  unique claims: {frame['claim_id'].nunique():,}")
print("sources:", frame.attrs.get("docs_path"), "|", frame.attrs.get("noisy_path"))

display(frame["surface"].value_counts().rename("count").to_frame())
display(
    frame.groupby(["document_type", "surface"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda d: d.sum(axis=1))
    .sort_values("total", ascending=False)
)
display(frame["split"].value_counts().rename("count").to_frame())
display(frame.head(3))

### Sample: typed vs handwriting/OCR text

In [ ]:
sample_id = frame.loc[frame["surface"] == SURFACE_TYPED, "record_id"].iloc[0]
pair = frame[frame["record_id"] == sample_id].set_index("surface")
print("record_id:", sample_id)
print("document_type:", pair.iloc[0]["document_type"])
print("\n--- TYPED ---\n")
print(pair.loc[SURFACE_TYPED, "text"][:700])
print("\n--- HANDWRITING / OCR ---\n")
print(pair.loc[SURFACE_HANDWRITING_OCR, "text"][:700])

## 4. Train Random Forest — document type classification

Features: TF-IDF unigrams + bigrams over the document text (typed and OCR surfaces together).

In [ ]:
train_df = frame[frame["split"] == "train"].reset_index(drop=True)
val_df = frame[frame["split"] == "val"].reset_index(drop=True)
test_df = frame[frame["split"] == "test"].reset_index(drop=True)

# Fit on train + val (classical baseline); hold out test for final prediction
fit_df = pd.concat([train_df, val_df], ignore_index=True)

doc_clf = build_document_type_pipeline(
    n_estimators=300,
    max_features=20000,
    ngram_range=(1, 2),
    random_state=42,
)
doc_clf.fit(fit_df["text"], fit_df["document_type"])

print(f"fitted on {len(fit_df):,} rows  |  classes: {list(doc_clf.classes_)}")
print(f"TF-IDF vocabulary size: {len(doc_clf.named_steps['tfidf'].vocabulary_):,}")

## 5. Evaluate on held-out test set

In [ ]:
doc_metrics = evaluate_classifier(
    doc_clf,
    test_df["text"],
    test_df["document_type"],
    labels=list(doc_clf.classes_),
)

print(f"Test accuracy : {doc_metrics['accuracy']:.4f}")
print(f"Macro F1      : {doc_metrics['macro_f1']:.4f}")
print(f"Weighted F1   : {doc_metrics['weighted_f1']:.4f}")
print(f"N test rows   : {doc_metrics['n']}")

report_df = pd.DataFrame(doc_metrics["classification_report"]).T
display(report_df.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(
    confusion_matrix=np.asarray(doc_metrics["confusion_matrix"]),
    display_labels=doc_metrics["labels"],
).plot(ax=ax, xticks_rotation=45, colorbar=True, cmap="Blues")
ax.set_title("Document-type Random Forest — test confusion matrix")
plt.tight_layout()
plt.show()

### Accuracy by surface (typed vs handwriting/OCR)

In [ ]:
surface_rows = []
for surface, group in test_df.groupby("surface"):
    m = evaluate_classifier(
        doc_clf, group["text"], group["document_type"], labels=list(doc_clf.classes_)
    )
    surface_rows.append(
        {
            "surface": surface,
            "n": m["n"],
            "accuracy": m["accuracy"],
            "macro_f1": m["macro_f1"],
        }
    )
surface_metrics = pd.DataFrame(surface_rows).sort_values("surface")
display(surface_metrics.round(4))

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.barplot(data=surface_metrics, x="surface", y="accuracy", hue="surface", ax=ax, legend=False)
ax.set_ylim(0, 1.05)
ax.set_title("Document-type accuracy by text surface")
ax.set_ylabel("accuracy")
plt.tight_layout()
plt.show()

## 6. Predictions

Generate class predictions + confidence for every test document.

In [ ]:
y_pred = doc_metrics["predictions"]
y_conf = doc_metrics["max_proba"]

pred_df = test_df[["record_id", "claim_id", "surface", "document_type"]].copy()
pred_df = pred_df.rename(columns={"document_type": "true_label"})
pred_df["predicted_label"] = y_pred
pred_df["confidence"] = y_conf
pred_df["correct"] = pred_df["true_label"] == pred_df["predicted_label"]

print("prediction accuracy:", round(float(pred_df["correct"].mean()), 4))
display(pred_df.head(10))

mistakes = pred_df[~pred_df["correct"]].sort_values("confidence", ascending=False)
print(f"\nmisclassified: {len(mistakes)} / {len(pred_df)}")
display(mistakes.head(10))

## 7. Top TF-IDF features (Random Forest importance)

In [ ]:
fi = top_tfidf_feature_importances(doc_clf, top_k=25)
display(fi)

fig, ax = plt.subplots(figsize=(8, 7))
sns.barplot(data=fi, y="feature", x="importance", color="#2a6f97", ax=ax)
ax.set_title("Top TF-IDF features by Random Forest importance")
ax.set_xlabel("mean decrease in impurity")
plt.tight_layout()
plt.show()

## 8. Secondary model — surface style (typed vs handwriting/OCR)

A second Random Forest predicts whether the text looks typed/clean or OCR/handwriting-noisy.

In [ ]:
surface_clf = build_document_type_pipeline(
    n_estimators=200,
    max_features=15000,
    ngram_range=(1, 2),
    random_state=42,
)
surface_clf.fit(fit_df["text"], fit_df["surface"])

surface_eval = evaluate_classifier(
    surface_clf,
    test_df["text"],
    test_df["surface"],
    labels=[SURFACE_TYPED, SURFACE_HANDWRITING_OCR],
)
print(f"Surface accuracy : {surface_eval['accuracy']:.4f}")
print(f"Surface macro F1 : {surface_eval['macro_f1']:.4f}")
display(pd.DataFrame(surface_eval["classification_report"]).T.round(3))

fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(
    confusion_matrix=np.asarray(surface_eval["confusion_matrix"]),
    display_labels=surface_eval["labels"],
).plot(ax=ax, colorbar=False, cmap="Greens")
ax.set_title("Surface classifier (typed vs handwriting/OCR)")
plt.tight_layout()
plt.show()

## 9. Multilayer train + WandB (official experiment)

Runs the same multilayer suite as `python -m src.classification.train_random_forest`:

capacity sweep (`shallow` / `balanced` / `deep` / `char_robust`) → best model →
surface head → typed/OCR slices → confidence/ECE → feature importance.

Artifacts land in `models/random_forest_classifier/` and metrics go to WandB under
namespaced keys (`data/`, `sweep/`, `best/`, `slice/`, `confidence/`, `interp/`).

In [ ]:
wb_settings = load_wandb_settings()
out_dir = train_rf_multilayer(
    ensure_data=False,
    presets=list(DEFAULT_PRESET_NAMES),
    wandb_settings=wb_settings,
    wandb_run_name="rf-notebook-multilayer",
)
print("multilayer artifacts ->", out_dir)
print(
    "WandB:",
    f"enabled={wb_settings.enabled}",
    f"mode={wb_settings.mode}",
    f"project={wb_settings.project}",
)
for name in (
    "sweep_results.json",
    "layer_diagnostics.json",
    "eval_metrics.json",
    "train_meta.json",
):
    path = out_dir / name
    print(f"  {name}: {'ok' if path.exists() else 'MISSING'}")

## 10. Quick inference helper

Predict document type for an arbitrary snippet (typed or OCR-like text).

In [ ]:
def predict_document(text: str, model=doc_clf) -> dict:
    label = model.predict([text])[0]
    proba = model.predict_proba([text])[0]
    conf = float(proba.max())
    ranking = sorted(
        zip(model.classes_, proba),
        key=lambda x: x[1],
        reverse=True,
    )
    return {
        "predicted_label": label,
        "confidence": conf,
        "top3": [(c, float(p)) for c, p in ranking[:3]],
    }


demo = """PROPERTY LOSS NOTICE
Claim Number: CLM-2026-111222
Date of Loss: 2025-11-03
Loss Type: water
Description of Loss: Pipe burst in upstairs bathroom; ceiling damage reported.
Reported By: Jordan Lee
"""

print(predict_document(demo))